# A9 Diffusion Auxiliary Experiments

Runs A9 conditional suffix diffusion with D3PM and forecast auxiliaries, then compares frozen/full fine-tuning and generation metrics.

In [ ]:
# Cell 1 - Setup
import copy
import json
import pathlib
import subprocess
import sys

import pandas as pd

WORKING_DIR = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
REPO_CANDIDATES = [WORKING_DIR / 'denoising-event-sequences', pathlib.Path.cwd(), pathlib.Path.cwd().parent]
REPO_DIR = next((p for p in REPO_CANDIDATES if (p / 'src').exists() and (p / 'scripts').exists()), REPO_CANDIDATES[0])
if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repository not found. Checked: {REPO_CANDIDATES}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--quiet'], check=True)
sys.path.insert(0, str(REPO_DIR))

try:
    import yaml
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyyaml', '--quiet'], check=True)
    import yaml

from src.utils.config import load_experiment_config


In [ ]:
# Cell 2 - Paths and runtime knobs
DATA_ROOT = WORKING_DIR / 'data'
PROCESSED_DIR = DATA_ROOT / 'processed' / 'rosbank_diffusion'
OUTPUT_DIR = WORKING_DIR / 'outputs' / 'diffusion_aux'
CONFIG_DIR = OUTPUT_DIR / 'configs'
LOG_DIR = OUTPUT_DIR / 'logs'

for path in [OUTPUT_DIR, CONFIG_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FAST_DEBUG = False
RUN_EXPERIMENTS = True
RUN_BASELINE_A9 = True
RUN_FROZEN_FINETUNE = True
RUN_FULL_FINETUNE = True
RUN_GENERATION = True

NUM_EPOCHS_PRETRAIN = 2 if FAST_DEBUG else 100
NUM_EPOCHS_FINETUNE = 2 if FAST_DEBUG else 20
BATCH_SIZE = 32 if FAST_DEBUG else 128
GEN_NUM_ENTITIES = 64 if FAST_DEBUG else 512
GEN_SUFFIX_LEN = 16

required = ['events.parquet', 'splits.json', 'preprocessor.pkl']
missing = [name for name in required if not (PROCESSED_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing processed artifacts in {PROCESSED_DIR}: {missing}')


In [ ]:
# Cell 3 - Base A9 config and experiment matrix
def deep_update(base: dict, override: dict) -> dict:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


base_config = load_experiment_config(
    base_path=str(REPO_DIR / 'configs/base.yaml'),
    ablation_path=str(REPO_DIR / 'configs/ablations/A9_conditional_generation.yaml'),
)

best_a9_overrides = {
    'data': {
        'max_seq_len': 512,
        'min_seq_len': 5,
        'event_type_col': 'MCC',
        'timestamp_col': 'TRDATETIME',
        'target_col': 'target_flag',
        'numerical_cols': ['amount'],
        'categorical_cols': ['channel_type', 'trx_category'],
        'group_col': 'cl_id',
        'truncation_pretrain': 'sliding_window',
        'truncation_eval': 'last_events',
        'amount_transform': 'robust_scaler',
        'amount_cols': ['amount'],
        'robust_scale_cols': ['amount'],
        'time_transform': 'none',
        'use_time_features': True,
        'time_feature_mode': 'derived_numeric',
        'min_vocab_count': 5,
    },
    'model': {
        'event_type_emb_dim': 128,
        'cat_emb_dim': 64,
        'num_projection_dim': 128,
        'time_projection_dim': 64,
        'hidden_dim': 512,
        'num_layers': 6,
        'num_heads': 8,
        'dim_feedforward': 2048,
        'dropout': 0.18,
        'activation': 'gelu',
        'max_seq_len': 512,
        'use_profile_token': True,
        'client_embedding_dim': 512,
    },
    'pooling': {
        'type': 'multi',
        'components': ['profile', 'gated_attention', 'mean', 'max', 'last'],
    },
    'diffusion': {
        'num_steps': 300,
        'timestep_sampling': 'uniform',
        'beta_start': 0.00003,
        'beta_end': 0.012,
        'discrete_mask_fraction': 0.60,
    },
    'generation': {
        'enabled': True,
        'mode': 'conditional_suffix',
        'suffix_len': 16,
        'prefix_min_ratio': 0.40,
        'prefix_max_ratio': 0.85,
        'sampler': 'ddim_lite',
        'num_sampling_steps': 50,
        'temperature_event_type': 1.05,
        'temperature_cat': 1.0,
        'top_k_event_type': 50,
        'continuous_update': 'x0_eps_blend',
    },
    'forecasting': {
        'enabled': False,
        'cut_min_ratio': 0.50,
        'cut_max_ratio': 0.80,
        'min_future_events': 2,
        'count_num_buckets': 6,
        'gap_num_buckets': 6,
        'alpha_forecast': 0.2,
        'pretrain_aux_enabled': False,
        'pretrain_aux_weight': 0.03,
        'finetune_aux_enabled': False,
    },
    'loss_calibration': {'enabled': False},
    'training': {
        'batch_size': BATCH_SIZE,
        'num_epochs_pretrain': NUM_EPOCHS_PRETRAIN,
        'num_epochs_finetune': NUM_EPOCHS_FINETUNE,
        'lr': 0.00015,
        'lr_encoder': 0.00001,
        'weight_decay': 0.03,
        'warmup_ratio': 0.05,
        'gradient_clip_val': 1.0,
        'mixed_precision': False,
        'early_stopping_patience': 5,
        'log_every_n_steps': 50,
    },
    'd3pm': {
        'enabled': False,
        'apply_to': ['event_type'],
        'transition': 'absorbing_uniform',
        'loss_weight_event_type_prev': 0.25,
        'use_prev_logits_for_sampling': True,
    },
}
deep_update(base_config, best_a9_overrides)

experiment_overrides = {
    'a9_baseline': {},
    'a9_d3pm_aux': {'d3pm': {'enabled': True}},
    'a9_forecast_aux': {'forecasting': {'enabled': True, 'pretrain_aux_enabled': True}},
    'a9_d3pm_forecast_aux': {
        'd3pm': {'enabled': True},
        'forecasting': {'enabled': True, 'pretrain_aux_enabled': True},
    },
}

experiment_configs = {}
for name, override in experiment_overrides.items():
    cfg = copy.deepcopy(base_config)
    deep_update(cfg, copy.deepcopy(override))
    if name == 'a9_baseline' and not RUN_BASELINE_A9:
        continue
    cfg_path = CONFIG_DIR / f'{name}.yaml'
    with cfg_path.open('w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    experiment_configs[name] = cfg_path
    print(f'{name}: {cfg_path}')


In [ ]:
# Cell 4 - Execution helpers
def run_logged(cmd: list[str], log_path: pathlib.Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Running:', ' '.join(cmd))
    with log_path.open('w') as log_file:
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        log_file.write(proc.stdout)
    print(proc.stdout[-4000:])
    print(f'Log saved: {log_path}')
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')


def load_json(path: pathlib.Path) -> dict:
    with path.open() as f:
        return json.load(f)


def load_metrics(path: pathlib.Path, prefix: str) -> dict:
    row = load_json(path)
    return {f'{prefix}_{k}': v for k, v in row.items() if isinstance(v, (int, float))}


In [ ]:
# Cell 5 - Run pretrain, fine-tune and generation for each experiment
rows = []

for exp_name, cfg_path in experiment_configs.items():
    exp_dir = OUTPUT_DIR / exp_name
    pretrain_dir = exp_dir / 'pretrain'
    frozen_dir = exp_dir / 'finetune_frozen'
    full_dir = exp_dir / 'finetune_full'
    generation_dir = exp_dir / 'generation'
    cfg_stem = cfg_path.stem

    pretrain_summary_path = pretrain_dir / 'metrics' / f'{cfg_stem}_pretrain_summary.json'
    if RUN_EXPERIMENTS:
        run_logged(
            [sys.executable, str(REPO_DIR / 'scripts/pretrain.py'), '--config', str(cfg_path), '--data-dir', str(PROCESSED_DIR), '--output-dir', str(pretrain_dir)],
            LOG_DIR / f'{exp_name}_pretrain.log',
        )
    assert pretrain_summary_path.exists(), f'Missing pretrain summary: {pretrain_summary_path}'
    pretrain_summary = load_json(pretrain_summary_path)
    ckpt_path = pathlib.Path(pretrain_summary['best_checkpoint'])
    assert ckpt_path.exists(), f'Missing checkpoint: {ckpt_path}'

    if RUN_EXPERIMENTS and RUN_FROZEN_FINETUNE:
        run_logged(
            [sys.executable, str(REPO_DIR / 'scripts/finetune.py'), '--config', str(cfg_path), '--pretrained-checkpoint', str(ckpt_path), '--data-dir', str(PROCESSED_DIR), '--output-dir', str(frozen_dir), '--frozen-encoder'],
            LOG_DIR / f'{exp_name}_finetune_frozen.log',
        )

    if RUN_EXPERIMENTS and RUN_FULL_FINETUNE:
        run_logged(
            [sys.executable, str(REPO_DIR / 'scripts/finetune.py'), '--config', str(cfg_path), '--pretrained-checkpoint', str(ckpt_path), '--data-dir', str(PROCESSED_DIR), '--output-dir', str(full_dir)],
            LOG_DIR / f'{exp_name}_finetune_full.log',
        )

    if RUN_EXPERIMENTS and RUN_GENERATION:
        run_logged(
            [sys.executable, str(REPO_DIR / 'scripts/generate_events.py'), '--config', str(cfg_path), '--checkpoint', str(ckpt_path), '--data-dir', str(PROCESSED_DIR), '--output-dir', str(generation_dir), '--split', 'test', '--num-entities', str(GEN_NUM_ENTITIES), '--num-samples', '1', '--suffix-len', str(GEN_SUFFIX_LEN)],
            LOG_DIR / f'{exp_name}_generation.log',
        )

    row = {'experiment': exp_name, 'config': str(cfg_path), 'checkpoint': str(ckpt_path)}
    frozen_metrics = frozen_dir / 'metrics' / f'{cfg_stem}_finetune_metrics.json'
    full_metrics = full_dir / 'metrics' / f'{cfg_stem}_finetune_metrics.json'
    generation_metrics = generation_dir / 'generation_metrics.json'
    if frozen_metrics.exists():
        row.update(load_metrics(frozen_metrics, 'frozen'))
    if full_metrics.exists():
        row.update(load_metrics(full_metrics, 'full'))
    if generation_metrics.exists():
        row.update(load_metrics(generation_metrics, 'generation'))
    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_path = OUTPUT_DIR / 'a9_aux_experiment_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved: {metrics_path}')
metrics_df


In [ ]:
# Cell 6 - Compare against checked-in DME results when available
comparison_rows = []
results_fin = REPO_DIR / 'results' / 'dme' / 'results_fin.csv'
if results_fin.exists():
    previous = pd.read_csv(results_fin)
    previous['source'] = 'results/dme'
    comparison_rows.append(previous)

current = metrics_df.copy()
current['source'] = 'current_aux'
comparison_rows.append(current)

comparison_df = pd.concat(comparison_rows, ignore_index=True, sort=False)
comparison_path = OUTPUT_DIR / 'a9_aux_vs_results.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f'Saved comparison: {comparison_path}')
display_cols = [c for c in ['source', 'experiment', 'pipeline', 'full_accuracy', 'full_macro_f1', 'full_roc_auc', 'full_pr_auc', 'full_balanced_accuracy', 'generation_event_type_accuracy', 'generation_event_type_top5_recall', 'generation_transition_js_divergence'] if c in comparison_df.columns]
comparison_df[display_cols].tail(20)
